# Physical Validation: Hardware Shot-Noise & The True Quantum Kernel
---
This notebook tests the physical viability of the Fourier PAC-learning framework on NISQ devices. Instead of using exact statevectors, we utilize the **Fast Emulator** (`execution_mode='emulator'`) to mathematically inject the $1/\sqrt{N}$ Binomial measurement noise expected from the QPU. 

This bypasses the classical CPU bottleneck of simulating deep Hadamard-test circuits while perfectly replicating the statistical outcome of running them on real hardware.

We will validate:
1. **$d=1$ (Homogeneous TFIM):** The `lasso` method's accuracy and robustness against $1/\sqrt{N}$ measurement noise.
2. **$d>1$ (Inhomogeneous TFIM):** The `kernel` method (True Quantum Overlap Kernel) completely bypassing the exponential feature space extraction while surviving measurement noise.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from qiskit.primitives import StatevectorSampler

from quantum_learning_dynamics import Experiment
from quantum_learning_dynamics.hamiltonians.tfim import TFIM, InhomogeneousTFIM
from quantum_learning_dynamics.observables.library import LocalMagnetization, StaggeredMagnetization
from quantum_learning_dynamics.features.engines import FeatureEngine

plt.rcParams.update({
    'figure.dpi': 120, 
    'axes.grid': True, 
    'grid.alpha': 0.3, 
    'font.family': 'sans-serif'
})


## 1. Feature Extraction under Shot Noise ($d=1$)
---
We first verify that our mathematical emulator perfectly reproduces the expected error scaling of a quantum device before running the full machine learning pipeline.

In [2]:
# Setup a simple d=1 model to verify the Hadamard Test mechanics
model_1d = TFIM(num_qubits=3)
observable_1d = LocalMagnetization(num_qubits=3, site=0) 
pauli = "IIZ"
x_graph = [(0, 1), (1, 2)] 
tau = 1.0
r_steps = 1
max_freq = 2 * r_steps * model_1d.num_qubits
rng = np.random.default_rng(42)

# 1. Exact Statevector Baseline (shots=None)
engine_exact = FeatureEngine(
    model=model_1d, trotter_order=1, execution_mode="emulator", 
    shots=None, sampler=None, rng=rng
)
b_sv = engine_exact._extract_emulator(x_graph, tau, r_steps, pauli, max_freq)

# 2. Shot-based Hardware Emulation (shots=8000)
shots = 8000
engine_hw = FeatureEngine(
    model=model_1d, trotter_order=1, execution_mode="hardware", 
    shots=shots, sampler=StatevectorSampler(), rng=rng  
)
b_hw = engine_hw._extract_emulator(x_graph, tau, r_steps, pauli, max_freq)

# 3. Compare Results
freqs = np.arange(-max_freq, max_freq + 1)

print(f"Comparing Extraction Methods (Shots = {shots})")
print(f"Expected RMS Error: {1/np.sqrt(shots):.4f}\n")
print(f"{'Freq':>5} | {'Statevector':>11} | {'Hardware':>11} | {'Diff':>6}")
print("-" * 43)

for f, sv, hw in zip(freqs, b_sv, b_hw):
    if abs(sv) > 1e-3 or abs(hw) > 1.5/np.sqrt(shots):
        print(f"{f:5d} | {sv:11.4f} | {hw:11.4f} | {abs(sv-hw):6.4f}")

rms_error = np.sqrt(np.mean((b_sv - b_hw)**2))
print("-" * 43)
print(f"Measured RMS Error: {rms_error:.4f}")

Comparing Extraction Methods (Shots = 8000)
Expected RMS Error: 0.0112

 Freq | Statevector |    Hardware |   Diff
-------------------------------------------
   -6 |      0.0000 |      0.0203 | 0.0202
   -2 |      0.5000 |      0.4925 | 0.0075
    1 |      0.0000 |     -0.0212 | 0.0212
    2 |      0.5000 |      0.5128 | 0.0128
-------------------------------------------
Measured RMS Error: 0.0119


In [3]:
# Run the full ML pipeline purely on shot-based features
exp_hw = Experiment(
    model=model_1d,
    observable=observable_1d,
    method="lasso",               
    execution_mode="hardware",    
    shots=5000,
    sampler=StatevectorSampler(),               
    tau=1.5,
    r_steps=2,
    lasso_alpha=0.001,  # Threshold the noise
    seed=42
)

result_hw = exp_hw.run(num_train=30, num_test=10)

print(f"--- Hardware PAC-Learning Results ({exp_hw.shots} shots/circuit) ---")
print(f"Trotter Generalization MSE: {result_hw.mse_trotter:.6f}")
print(f"Exact Physics MSE:          {result_hw.mse_exact:.6f}\n")

print("Predictions on Unseen Topologies:")
for i in range(5):
    print(f"Test {i}: True = {result_hw.y_true_trotter[i]:+.4f} | Pred = {result_hw.y_pred[i]:+.4f}")

--- Hardware PAC-Learning Results (5000 shots/circuit) ---
Trotter Generalization MSE: 0.000774
Exact Physics MSE:          0.031425

Predictions on Unseen Topologies:
Test 0: True = -0.8468 | Pred = -0.7849
Test 1: True = +0.0113 | Pred = +0.0087
Test 2: True = +0.0113 | Pred = -0.0217
Test 3: True = -0.8468 | Pred = -0.8413
Test 4: True = +0.0113 | Pred = -0.0306


## 2. True Quantum Overlap Kernel under Shot Noise ($d>1$)
---
For inhomogeneous systems ($d>1$), extracting the Fourier tensor is physically unscalable. We use the **True Quantum Kernel** to compute the $\mathcal{O}(M^2)$ pairwise Gram matrix directly.

By using `execution_mode='emulator'` and `shots=4000`, the `KernelEngine` builds exact statevectors, computes the pairwise overlaps instantly, and corrupts them with Binomial noise to prove that Kernel Ridge Regression survives physical QPU noise.

In [4]:
# Setup a d>1 model
model_nd = InhomogeneousTFIM(num_qubits=3)
obs_nd = StaggeredMagnetization(num_qubits=3)

exp_kernel = Experiment(
    model=model_nd,
    observable=obs_nd,
    method="kernel",              # Overlap Circuit (Bypasses Tensor)
    execution_mode="hardware",    # Fast O(M) Statevector caching
    shots=4000,                   # Inject Binomial Hardware Noise
    tau=1.0,
    r_steps=2,
    kernel_alpha=0.05,            # Higher Ridge penalty for noisy Gram matrix
    seed=42
)

# This would take ~1 hour on Qiskit Sampler, but completes in < 1 second here!
t0 = time.time()
result_kernel = exp_kernel.run(num_train=20, num_test=10)
exec_time = time.time() - t0

print(f"--- d>1 True Quantum Kernel Results ({exp_kernel.shots} shots) ---")
print(f"Execution Time:             {exec_time:.2f} seconds")
print(f"Trotter Generalization MSE: {result_kernel.mse_trotter:.6f}")
print(f"Exact Physics MSE:          {result_kernel.mse_exact:.6f}")

: 